# Tugas Besar 2 IF3270 — CNN Image Classification
Dataset: Intel Image Classification (~25.000 gambar, 6 kelas)
**Google Colab Version**

## 0. Setup Google Colab

In [ ]:
# mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

# sesuaikan path ini dengan lokasi file di Drive kamu
DRIVE_ROOT      = '/content/drive/MyDrive'
ZIP_PATH        = '/content/drive/MyDrive/Tubes2ML/PixelToCaption.zip'
INTEL_DATA_PATH = '/content/drive/MyDrive/Tubes2ML/intel'

# verifikasi file ada
print(f'ZIP ada    : {os.path.exists(ZIP_PATH)}')
print(f'Intel ada  : {os.path.exists(INTEL_DATA_PATH)}')

In [ ]:
# extract PixelToCaption.zip ke /content
import zipfile

REPO_DIR = '/content/PixelToCaption'

if not os.path.exists(REPO_DIR):
    print('Mengekstrak PixelToCaption.zip...')
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        z.extractall('/content')
    print('Ekstraksi selesai')
else:
    print('Repo sudah ada, skip ekstraksi')

print(f'Isi repo: {os.listdir(REPO_DIR)}')

In [ ]:
# buat __init__.py yang diperlukan untuk import
open(os.path.join(REPO_DIR, 'src', '__init__.py'), 'a').close()
open(os.path.join(REPO_DIR, 'src', 'cnn', '__init__.py'), 'a').close()
print('__init__.py sudah dibuat')

In [ ]:
# install dependencies yang belum ada di Colab
!pip install scikit-learn tqdm -q

In [ ]:
# set path repo agar bisa import
import sys
sys.path.insert(0, REPO_DIR)

# set DATA_DIR dan WEIGHTS_DIR
DATA_DIR    = INTEL_DATA_PATH
WEIGHTS_DIR = os.path.join(DRIVE_ROOT, 'weights', 'cnn')  # simpan ke Drive agar tidak hilang
os.makedirs(WEIGHTS_DIR, exist_ok=True)

print(f'REPO_DIR   : {REPO_DIR}')
print(f'DATA_DIR   : {DATA_DIR}')
print(f'WEIGHTS_DIR: {WEIGHTS_DIR}')

In [ ]:
# import semua yang diperlukan
import numpy as np
import matplotlib.pyplot as plt
import glob
import tensorflow as tf
from tensorflow import keras

from src.cnn.train import (
    load_dataset, build_cnn_model, build_locally_connected_model,
    run_experiments, CLASS_NAMES, IMG_SIZE, EPOCHS
)
from src.cnn.cnn import CNNScratch
from src.cnn.layers import (
    Conv2DLayer, MaxPooling2DLayer, AveragePooling2DLayer,
    GlobalAveragePooling2DLayer, FlattenLayer,
    LocallyConnected2DLayer, ActivationLayer
)
from src.shared.dense import DenseLayer
from src.shared.image_utils import load_image_cnn, load_batch_cnn, extract_features_cnn
from src.cnn.utils import (
    evaluate_keras_model, evaluate_scratch_model,
    plot_history, plot_f1_comparison
)

# override WEIGHTS_DIR di train.py agar bobot disimpan ke Drive
import src.cnn.train as train_module
train_module.WEIGHTS_DIR = WEIGHTS_DIR

print('Import selesai')
print(f'GPU tersedia: {len(tf.config.list_physical_devices("GPU")) > 0}')
print(f'Kelas: {CLASS_NAMES}')

## 1. Load Dataset

In [ ]:
train_ds, val_ds = load_dataset(DATA_DIR)

# kumpulkan test set ke numpy untuk evaluasi scratch
X_test, y_test = [], []
for X_batch, y_batch in val_ds:
    X_test.append(X_batch.numpy())
    y_test.append(y_batch.numpy())
X_test = np.concatenate(X_test)
y_test = np.concatenate(y_test)

print(f'Test set shape  : {X_test.shape}')
print(f'Label shape     : {y_test.shape}')
print(f'Distribusi label: {dict(zip(*np.unique(y_test, return_counts=True)))}')

In [ ]:
# tampilkan sample gambar per kelas
fig, axes = plt.subplots(1, 6, figsize=(18, 3))
for cls_idx, ax in enumerate(axes):
    mask = y_test == cls_idx
    img  = X_test[mask][0]
    ax.imshow(img)
    ax.set_title(CLASS_NAMES[cls_idx])
    ax.axis('off')
plt.suptitle('Sample Gambar per Kelas')
plt.tight_layout()
plt.show()

## 2. Training Semua Variasi Conv2D (16 Arsitektur)
> **SKIP cell ini kalau bobot sudah ada di Drive**

In [ ]:
# cek bobot yang sudah ada
existing = [f for f in os.listdir(WEIGHTS_DIR) if f.endswith('.weights.h5')]
print(f'Bobot yang sudah ada ({len(existing)}): {existing}')

In [ ]:
# variasi: 2 jumlah layer x 2 filter x 2 kernel x 2 pooling = 16 arsitektur
# bobot disimpan langsung ke Drive agar tidak hilang saat session disconnect
results = run_experiments(DATA_DIR)
print('Semua eksperimen selesai')

## 3. Definisi Konfigurasi

In [ ]:
filter_variants = {
    'small': {2: [32, 64],    3: [32, 64, 64]},
    'large': {2: [64, 128],   3: [64, 128, 128]}
}
kernel_variants = {
    'k3': {2: [3, 3],   3: [3, 3, 3]},
    'k5': {2: [5, 5],   3: [5, 5, 5]}
}

all_configs = []
for n_layers in [2, 3]:
    for fkey, f_dict in filter_variants.items():
        for kkey, k_dict in kernel_variants.items():
            for pool in ['max', 'average']:
                all_configs.append({
                    'name':            f'conv{n_layers}_{fkey}_{kkey}_{pool}',
                    'num_conv_layers': n_layers,
                    'filters':         f_dict[n_layers],
                    'kernel_sizes':    k_dict[n_layers],
                    'pooling_type':    pool
                })

print(f'Total konfigurasi: {len(all_configs)}')

## 4. Evaluasi Semua Variasi — Macro F1-Score

In [ ]:
all_results = {}

for cfg in all_configs:
    w_path = os.path.join(WEIGHTS_DIR, f"{cfg['name']}.weights.h5")
    if not os.path.exists(w_path):
        print(f"SKIP {cfg['name']} — bobot tidak ditemukan")
        continue

    model = build_cnn_model(
        cfg['num_conv_layers'], cfg['filters'],
        cfg['kernel_sizes'],    cfg['pooling_type']
    )
    model.load_weights(w_path)

    res = evaluate_keras_model(model, X_test, y_test, CLASS_NAMES)
    all_results[cfg['name']] = {**res, 'config': cfg}
    print(f"{cfg['name']:<35} macro F1 = {res['macro_f1']:.4f}")

print(f'\nTotal model dievaluasi: {len(all_results)}')

## 5. Analisis Pengaruh Jumlah Conv Layer

In [ ]:
labels, scores = [], []
for n in [2, 3]:
    name = f'conv{n}_large_k3_max'
    labels.append(f'{n} layers')
    scores.append(all_results[name]['macro_f1'])

plot_f1_comparison(labels, scores, title='Pengaruh Jumlah Conv Layer — Macro F1')
print(f'Terbaik: {labels[np.argmax(scores)]} (F1={max(scores):.4f})')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for n in [2, 3]:
    name = f'conv{n}_large_k3_max'
    h    = results[name]['history']
    axes[0].plot(h['loss'],     label=f'{n} layers')
    axes[1].plot(h['val_loss'], label=f'{n} layers')

axes[0].set_title('Train Loss'); axes[0].set_xlabel('Epoch')
axes[1].set_title('Val Loss');   axes[1].set_xlabel('Epoch')
for ax in axes: ax.legend(); ax.grid(True)
plt.suptitle('Pengaruh Jumlah Conv Layer')
plt.tight_layout(); plt.show()

## 6. Analisis Pengaruh Jumlah Filter

In [ ]:
filter_labels = {'small': '[32,64]', 'large': '[64,128]'}
labels, scores = [], []
for fkey in ['small', 'large']:
    name = f'conv2_{fkey}_k3_max'
    labels.append(filter_labels[fkey])
    scores.append(all_results[name]['macro_f1'])

plot_f1_comparison(labels, scores, title='Pengaruh Jumlah Filter — Macro F1')
print(f'Terbaik: {labels[np.argmax(scores)]} (F1={max(scores):.4f})')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for fkey in ['small', 'large']:
    name = f'conv2_{fkey}_k3_max'
    h    = results[name]['history']
    axes[0].plot(h['loss'],     label=filter_labels[fkey])
    axes[1].plot(h['val_loss'], label=filter_labels[fkey])

axes[0].set_title('Train Loss'); axes[0].set_xlabel('Epoch')
axes[1].set_title('Val Loss');   axes[1].set_xlabel('Epoch')
for ax in axes: ax.legend(); ax.grid(True)
plt.suptitle('Pengaruh Jumlah Filter')
plt.tight_layout(); plt.show()

## 7. Analisis Pengaruh Ukuran Filter

In [ ]:
kernel_labels = {'k3': '3x3', 'k5': '5x5'}
labels, scores = [], []
for kkey in ['k3', 'k5']:
    name = f'conv2_large_{kkey}_max'
    labels.append(kernel_labels[kkey])
    scores.append(all_results[name]['macro_f1'])

plot_f1_comparison(labels, scores, title='Pengaruh Ukuran Filter — Macro F1')
print(f'Terbaik: {labels[np.argmax(scores)]} (F1={max(scores):.4f})')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for kkey in ['k3', 'k5']:
    name = f'conv2_large_{kkey}_max'
    h    = results[name]['history']
    axes[0].plot(h['loss'],     label=kernel_labels[kkey])
    axes[1].plot(h['val_loss'], label=kernel_labels[kkey])

axes[0].set_title('Train Loss'); axes[0].set_xlabel('Epoch')
axes[1].set_title('Val Loss');   axes[1].set_xlabel('Epoch')
for ax in axes: ax.legend(); ax.grid(True)
plt.suptitle('Pengaruh Ukuran Filter')
plt.tight_layout(); plt.show()

## 8. Analisis Pengaruh Jenis Pooling

In [ ]:
labels, scores = [], []
for pool in ['max', 'average']:
    name = f'conv2_large_k3_{pool}'
    labels.append(pool)
    scores.append(all_results[name]['macro_f1'])

plot_f1_comparison(labels, scores, title='Pengaruh Jenis Pooling — Macro F1')
print(f'Terbaik: {labels[np.argmax(scores)]} (F1={max(scores):.4f})')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for pool in ['max', 'average']:
    name = f'conv2_large_k3_{pool}'
    h    = results[name]['history']
    axes[0].plot(h['loss'],     label=pool)
    axes[1].plot(h['val_loss'], label=pool)

axes[0].set_title('Train Loss'); axes[0].set_xlabel('Epoch')
axes[1].set_title('Val Loss');   axes[1].set_xlabel('Epoch')
for ax in axes: ax.legend(); ax.grid(True)
plt.suptitle('Pengaruh Jenis Pooling')
plt.tight_layout(); plt.show()

## 9. Rekap Semua 16 Arsitektur & Pilih Terbaik

In [ ]:
sorted_results = sorted(all_results.items(), key=lambda x: x[1]['macro_f1'], reverse=True)

print(f"{'Nama':<35} {'Macro F1':>10}")
print('-' * 47)
for name, res in sorted_results:
    print(f"{name:<35} {res['macro_f1']:>10.4f}")

BEST_NAME   = sorted_results[0][0]
BEST_CONFIG = sorted_results[0][1]['config']
print(f'\nArsitektur terbaik : {BEST_NAME}')
print(f'Filters            : {BEST_CONFIG["filters"]}')
print(f'Kernel sizes       : {BEST_CONFIG["kernel_sizes"]}')
print(f'Pooling            : {BEST_CONFIG["pooling_type"]}')
print(f'Num layers         : {BEST_CONFIG["num_conv_layers"]}')

In [ ]:
all_labels = [name for name, _ in sorted_results]
all_scores = [res['macro_f1'] for _, res in sorted_results]
plot_f1_comparison(all_labels, all_scores, title='Macro F1 — Semua 16 Arsitektur Conv2D')

## 10. Feature Extraction menggunakan Arsitektur Terbaik

In [ ]:
# kumpulkan semua path gambar train dan test
train_paths = glob.glob(
    os.path.join(DATA_DIR, 'seg_train', 'seg_train', '**', '*.jpg'),
    recursive=True
)
test_paths = glob.glob(
    os.path.join(DATA_DIR, 'seg_test', 'seg_test', '**', '*.jpg'),
    recursive=True
)

print(f'Train images: {len(train_paths)}')
print(f'Test images : {len(test_paths)}')

In [ ]:
# load arsitektur terbaik sebagai encoder (frozen)
encoder = build_cnn_model(
    BEST_CONFIG['num_conv_layers'],
    BEST_CONFIG['filters'],
    BEST_CONFIG['kernel_sizes'],
    BEST_CONFIG['pooling_type']
)
encoder.load_weights(os.path.join(WEIGHTS_DIR, f'{BEST_NAME}.weights.h5'))
encoder.trainable = False

# simpan features ke Drive agar tidak hilang
FEATURES_DIR = os.path.join(DRIVE_ROOT, 'features')
os.makedirs(FEATURES_DIR, exist_ok=True)

# ekstraksi feature train
extract_features_cnn(
    paths       = train_paths,
    output_path = os.path.join(FEATURES_DIR, 'features_train.npy'),
    encoder     = encoder,
    batch_size  = 64,
    target_dim  = (150, 150)
)

# ekstraksi feature test
extract_features_cnn(
    paths       = test_paths,
    output_path = os.path.join(FEATURES_DIR, 'features_test.npy'),
    encoder     = encoder,
    batch_size  = 64,
    target_dim  = (150, 150)
)

In [ ]:
# verifikasi hasil feature extraction
features_train = np.load(
    os.path.join(FEATURES_DIR, 'features_train.npy'), allow_pickle=True
).item()
features_test  = np.load(
    os.path.join(FEATURES_DIR, 'features_test.npy'), allow_pickle=True
).item()

sample_key = list(features_train.keys())[0]
print(f'Jumlah fitur train  : {len(features_train)}')
print(f'Jumlah fitur test   : {len(features_test)}')
print(f'Shape feature vector: {features_train[sample_key].shape}')

## 11. Perbandingan Keras vs From Scratch (Shared Parameter / Conv2D)

In [ ]:
# load arsitektur terbaik
best_model = build_cnn_model(
    BEST_CONFIG['num_conv_layers'],
    BEST_CONFIG['filters'],
    BEST_CONFIG['kernel_sizes'],
    BEST_CONFIG['pooling_type']
)
best_model.load_weights(os.path.join(WEIGHTS_DIR, f'{BEST_NAME}.weights.h5'))
best_model.summary()

In [ ]:
# build scratch dari arsitektur terbaik
# kernel+bias dari Keras, activation dipisah pakai ActivationLayer
n_layers  = BEST_CONFIG['num_conv_layers']
best_pool = BEST_CONFIG['pooling_type']

scratch_layers = []
layer_idx = 1  # skip InputLayer di index 0

for i in range(n_layers):
    # Conv2D: padding='same', strides=(1,1), activation='relu'
    # sama persis dengan yang didefinisikan di build_cnn_model
    scratch_layers.append(
        Conv2DLayer(
            keras_layer = best_model.layers[layer_idx],
            activation  = 'relu',
            strides     = (1, 1),
            padding     = 'same'
        )
    )
    layer_idx += 1

    # Pooling: pool_size=(2,2) sama persis dengan build_cnn_model
    if best_pool == 'max':
        scratch_layers.append(MaxPooling2DLayer(pool_size=(2,2), strides=(2,2)))
    else:
        scratch_layers.append(AveragePooling2DLayer(pool_size=(2,2), strides=(2,2)))
    layer_idx += 1

# GlobalAveragePooling
scratch_layers.append(GlobalAveragePooling2DLayer())
layer_idx += 1

# Dense relu — aktivasi dipisah pakai ActivationLayer
scratch_layers.append(DenseLayer(keras_layer=best_model.layers[layer_idx]))
scratch_layers.append(ActivationLayer('relu'))
layer_idx += 1

# Dropout — skip
layer_idx += 1

# Dense softmax — aktivasi dipisah
scratch_layers.append(DenseLayer(keras_layer=best_model.layers[layer_idx]))
scratch_layers.append(ActivationLayer('softmax'))

scratch_model = CNNScratch(scratch_layers)
print(f'Jumlah layer scratch: {len(scratch_model.layers)}')

In [ ]:
# evaluasi Keras vs Scratch — pakai subset karena scratch lambat
N_EVAL = 200

keras_res   = evaluate_keras_model(best_model,     X_test[:N_EVAL], y_test[:N_EVAL], CLASS_NAMES)
scratch_res = evaluate_scratch_model(scratch_model, X_test[:N_EVAL], y_test[:N_EVAL], CLASS_NAMES)

print(f'Keras   macro F1 : {keras_res["macro_f1"]:.4f}')
print(f'Scratch macro F1 : {scratch_res["macro_f1"]:.4f}')
print(f'Selisih          : {abs(keras_res["macro_f1"] - scratch_res["macro_f1"]):.6f}')
print(f'Prediksi berbeda : {np.sum(keras_res["y_pred"] != scratch_res["y_pred"])} dari {N_EVAL}')

In [ ]:
plot_f1_comparison(
    ['Keras', 'From Scratch'],
    [keras_res['macro_f1'], scratch_res['macro_f1']],
    title='Conv2D — Keras vs From Scratch'
)
print('=== Keras ===')
print(keras_res['report'])
print('=== From Scratch ===')
print(scratch_res['report'])

## 12. Training LocallyConnected2D (Non-Shared Parameter)
> **SKIP kalau sudah pernah ditraining**

In [ ]:
# training LocallyConnected dengan config arsitektur terbaik
lc_model = build_locally_connected_model(
    filters      = BEST_CONFIG['filters'],
    kernel_sizes = BEST_CONFIG['kernel_sizes'],
    pooling_type = BEST_CONFIG['pooling_type']
)
lc_model.summary()

lc_history = lc_model.fit(
    train_ds, validation_data=val_ds,
    epochs=EPOCHS, verbose=1
)

# simpan ke Drive
lc_w_path = os.path.join(WEIGHTS_DIR, 'locally_connected_best.weights.h5')
lc_model.save_weights(lc_w_path)
print(f'Bobot LC disimpan: {lc_w_path}')

## 13. Perbandingan Shared vs Non-Shared

In [ ]:
# load LC dari Drive
lc_model = build_locally_connected_model(
    filters      = BEST_CONFIG['filters'],
    kernel_sizes = BEST_CONFIG['kernel_sizes'],
    pooling_type = BEST_CONFIG['pooling_type']
)
lc_model.load_weights(os.path.join(WEIGHTS_DIR, 'locally_connected_best.weights.h5'))

# evaluasi
shared_res    = evaluate_keras_model(best_model, X_test, y_test, CLASS_NAMES)
nonshared_res = evaluate_keras_model(lc_model,   X_test, y_test, CLASS_NAMES)

print(f'Conv2D (shared)               F1: {shared_res["macro_f1"]:.4f}')
print(f'LocallyConnected (non-shared) F1: {nonshared_res["macro_f1"]:.4f}')
print(f'\nParameter Conv2D           : {best_model.count_params():,}')
print(f'Parameter LocallyConnected  : {lc_model.count_params():,}')

In [ ]:
# plot loss shared vs non-shared
best_history = results[BEST_NAME]['history']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(best_history['loss'],           label='Conv2D (shared)')
axes[0].plot(lc_history.history['loss'],     label='LocallyConnected')
axes[1].plot(best_history['val_loss'],        label='Conv2D (shared)')
axes[1].plot(lc_history.history['val_loss'], label='LocallyConnected')

axes[0].set_title('Train Loss'); axes[0].set_xlabel('Epoch')
axes[1].set_title('Val Loss');   axes[1].set_xlabel('Epoch')
for ax in axes: ax.legend(); ax.grid(True)
plt.suptitle('Shared vs Non-Shared')
plt.tight_layout(); plt.show()

plot_f1_comparison(
    ['Conv2D\n(shared)', 'LocallyConnected\n(non-shared)'],
    [shared_res['macro_f1'], nonshared_res['macro_f1']],
    title='Shared vs Non-Shared — Macro F1'
)

## 14. From Scratch — LocallyConnected2D

In [ ]:
# build scratch LocallyConnected
# LocallyConnected di Keras default padding='valid'
lc_scratch_layers = []
layer_idx = 1

for i in range(len(BEST_CONFIG['filters'])):
    lc_scratch_layers.append(
        LocallyConnected2DLayer(
            keras_layer = lc_model.layers[layer_idx],
            activation  = 'relu',
            strides     = (1, 1),
            padding     = 'valid',
            kernel_size = (BEST_CONFIG['kernel_sizes'][i], BEST_CONFIG['kernel_sizes'][i])
        )
    )
    layer_idx += 1

    if BEST_CONFIG['pooling_type'] == 'max':
        lc_scratch_layers.append(MaxPooling2DLayer(pool_size=(2,2), strides=(2,2)))
    else:
        lc_scratch_layers.append(AveragePooling2DLayer(pool_size=(2,2), strides=(2,2)))
    layer_idx += 1

# Flatten
lc_scratch_layers.append(FlattenLayer())
layer_idx += 1

# Dense relu — aktivasi dipisah
lc_scratch_layers.append(DenseLayer(keras_layer=lc_model.layers[layer_idx]))
lc_scratch_layers.append(ActivationLayer('relu'))
layer_idx += 1

# Dropout — skip
layer_idx += 1

# Dense softmax — aktivasi dipisah
lc_scratch_layers.append(DenseLayer(keras_layer=lc_model.layers[layer_idx]))
lc_scratch_layers.append(ActivationLayer('softmax'))

lc_scratch = CNNScratch(lc_scratch_layers)
print(f'Jumlah layer LC scratch: {len(lc_scratch.layers)}')

In [ ]:
lc_keras_res   = evaluate_keras_model(lc_model,    X_test[:N_EVAL], y_test[:N_EVAL], CLASS_NAMES)
lc_scratch_res = evaluate_scratch_model(lc_scratch, X_test[:N_EVAL], y_test[:N_EVAL], CLASS_NAMES)

print(f'LC Keras   macro F1 : {lc_keras_res["macro_f1"]:.4f}')
print(f'LC Scratch macro F1 : {lc_scratch_res["macro_f1"]:.4f}')
print(f'Selisih             : {abs(lc_keras_res["macro_f1"] - lc_scratch_res["macro_f1"]):.6f}')
print(f'Prediksi berbeda    : {np.sum(lc_keras_res["y_pred"] != lc_scratch_res["y_pred"])} dari {N_EVAL}')

## 15. Rekap Akhir Semua Perbandingan

In [ ]:
print('='*55)
print(f"{'Model':<40} {'Macro F1':>10}")
print('='*55)
print(f"{'Conv2D terbaik (Keras)':<40} {shared_res['macro_f1']:>10.4f}")
print(f"{'Conv2D terbaik (Scratch)':<40} {scratch_res['macro_f1']:>10.4f}")
print(f"{'LocallyConnected (Keras)':<40} {nonshared_res['macro_f1']:>10.4f}")
print(f"{'LocallyConnected (Scratch)':<40} {lc_scratch_res['macro_f1']:>10.4f}")
print('='*55)
print(f"\nParameter Conv2D           : {best_model.count_params():,}")
print(f"Parameter LocallyConnected  : {lc_model.count_params():,}")

In [ ]:
plot_f1_comparison(
    ['Conv2D\nKeras', 'Conv2D\nScratch', 'LC\nKeras', 'LC\nScratch'],
    [
        shared_res['macro_f1'],
        scratch_res['macro_f1'],
        nonshared_res['macro_f1'],
        lc_scratch_res['macro_f1']
    ],
    title='Rekap Akhir — Macro F1 Semua Model'
)

## 16. BONUS

In [ ]:
# ── BONUS 1: Visualisasi Feature Maps ──────────────────────────
import matplotlib.cm as cm

def get_feature_maps(model, img, layer_names=None):
    # buat model yang output tiap Conv2D layer
    if layer_names is None:
        layer_names = [l.name for l in model.layers if 'conv2d' in l.name]

    feature_model = keras.Model(
        inputs  = model.input,
        outputs = [model.get_layer(name).output for name in layer_names]
    )
    # tambah batch dim
    img_batch = img[np.newaxis, ...]
    outputs   = feature_model.predict(img_batch, verbose=0)

    # kalau hanya 1 layer, wrap ke list
    if len(layer_names) == 1:
        outputs = [outputs]

    return layer_names, outputs


def visualize_feature_maps(model, img, n_filters=8):
    layer_names, outputs = get_feature_maps(model, img)

    for name, fmap in zip(layer_names, outputs):
        fmap   = fmap[0]          # hapus batch dim: (H, W, C)
        n_show = min(n_filters, fmap.shape[-1])

        fig, axes = plt.subplots(1, n_show, figsize=(2*n_show, 2.5))
        fig.suptitle(f'Feature Maps — {name}', fontsize=12)

        for i, ax in enumerate(axes):
            channel = fmap[:, :, i]
            # normalisasi ke [0, 1]
            channel = (channel - channel.min()) / (channel.max() - channel.min() + 1e-8)
            ax.imshow(channel, cmap='viridis')
            ax.set_title(f'Filter {i}', fontsize=8)
            ax.axis('off')

        plt.tight_layout()
        plt.show()

In [ ]:
def grad_cam(model, img, class_idx=None, last_conv_layer_name=None):
    # cari nama layer Conv2D terakhir kalau tidak dispecify
    if last_conv_layer_name is None:
        for layer in reversed(model.layers):
            if 'conv2d' in layer.name:
                last_conv_layer_name = layer.name
                break

    # buat model: input → [conv output, final output]
    grad_model = keras.Model(
        inputs  = model.input,
        outputs = [
            model.get_layer(last_conv_layer_name).output,
            model.output
        ]
    )

    img_batch = img[np.newaxis, ...].astype(np.float32)

    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_batch)
        if class_idx is None:
            class_idx = tf.argmax(predictions[0])
        loss = predictions[:, class_idx]

    # hitung gradient output kelas terhadap conv layer terakhir
    grads      = tape.gradient(loss, conv_outputs)        # (1, H, W, C)
    grads      = grads[0]                                 # (H, W, C)
    conv_out   = conv_outputs[0].numpy()                  # (H, W, C)

    # global average gradient per channel → bobot kepentingan
    weights    = np.mean(grads.numpy(), axis=(0, 1))      # (C,)

    # weighted sum feature map
    cam        = np.zeros(conv_out.shape[:2], dtype=np.float32)
    for i, w in enumerate(weights):
        cam   += w * conv_out[:, :, i]

    # relu → hanya region positif yang berpengaruh
    cam        = np.maximum(cam, 0)

    # normalisasi ke [0, 1]
    cam        = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)

    # resize ke ukuran gambar asli
    import cv2
    cam_resized = cv2.resize(cam, (img.shape[1], img.shape[0]))

    return cam_resized, int(class_idx)


def overlay_grad_cam(img, cam, alpha=0.4):
    # konversi heatmap ke colormap
    heatmap = np.uint8(255 * cam)
    heatmap = cm.jet(heatmap)[:, :, :3]  # ambil RGB saja

    # overlay ke gambar asli
    overlay = heatmap * alpha + img * (1 - alpha)
    overlay = np.clip(overlay, 0, 1)
    return overlay


def visualize_grad_cam(model, images, y_true, class_names, n_samples=5):
    # pilih sample dengan score tinggi, sedang, rendah
    y_pred_proba = model.predict(images, verbose=0)
    y_pred       = np.argmax(y_pred_proba, axis=1)
    correct_mask = y_pred == y_true
    confidence   = np.max(y_pred_proba, axis=1)

    # sort by confidence
    sorted_idx   = np.argsort(confidence)[::-1]
    # ambil tinggi, sedang, rendah
    high_idx     = sorted_idx[:n_samples]
    mid_idx      = sorted_idx[len(sorted_idx)//2 : len(sorted_idx)//2 + n_samples]
    low_idx      = sorted_idx[-n_samples:]

    for label, idxs in [('High Confidence', high_idx),
                         ('Mid Confidence',  mid_idx),
                         ('Low Confidence',  low_idx)]:
        fig, axes = plt.subplots(n_samples, 3, figsize=(12, 3*n_samples))
        fig.suptitle(f'Grad-CAM — {label}', fontsize=13)

        for row, idx in enumerate(idxs):
            img       = images[idx]
            cam, pred = grad_cam(model, img)
            overlay   = overlay_grad_cam(img, cam)

            axes[row, 0].imshow(img)
            axes[row, 0].set_title(f'Original\nTrue: {class_names[y_true[idx]]}')
            axes[row, 0].axis('off')

            axes[row, 1].imshow(cam, cmap='jet')
            axes[row, 1].set_title(f'Grad-CAM\nPred: {class_names[pred]}')
            axes[row, 1].axis('off')

            axes[row, 2].imshow(overlay)
            axes[row, 2].set_title(f'Overlay\nConf: {confidence[idx]:.2f}')
            axes[row, 2].axis('off')

        plt.tight_layout()
        plt.show()

In [ ]:
!pip install opencv-python-headless -q

# jalankan visualisasi feature maps pada 1 gambar contoh
sample_img = X_test[0]
print(f'Visualisasi feature maps untuk: {CLASS_NAMES[y_test[0]]}')
visualize_feature_maps(best_model, sample_img, n_filters=8)

In [ ]:
# jalankan Grad-CAM
print('Grad-CAM visualization...')
visualize_grad_cam(best_model, X_test[:50], y_test[:50], CLASS_NAMES, n_samples=3)

In [ ]:
# ── BONUS 2: Test Batch Inference ──────────────────────────────
print('Test batch inference from scratch...')

# test dengan berbagai batch_size
for bs in [1, 8, 32]:
    preds = scratch_model.predict_batch(X_test[:32], batch_size=bs)
    print(f'batch_size={bs:2d} → predictions shape: {preds.shape}, sample: {preds[:5]}')

# bandingkan hasil batch vs single
pred_batch  = scratch_model.predict_batch(X_test[:10], batch_size=10)
pred_single = np.array([scratch_model.predict(X_test[i]) for i in range(10)])

print(f'\nBatch  predictions : {pred_batch}')
print(f'Single predictions : {pred_single}')
print(f'Sama persis        : {np.all(pred_batch == pred_single)}')

In [ ]:
# ── BONUS 3: Test Backward Propagation ─────────────────────────
print('Test backward propagation...')

# gradient check sederhana — bandingkan numerical vs analytic gradient
def numerical_gradient(layer, x, eps=1e-5):
    # hitung gradient numerik (finite difference)
    out_plus  = layer.forward(x + eps)
    out_minus = layer.forward(x - eps)
    return (out_plus - out_minus) / (2 * eps)

# test Conv2DLayer backward
test_img   = X_test[0:2]  # batch 2 gambar
kernel_test = np.random.randn(3, 3, 3, 8).astype(np.float32) * 0.1
bias_test   = np.zeros(8, dtype=np.float32)
conv_test   = Conv2DLayer(kernel=kernel_test, bias=bias_test,
                           activation='relu', padding='same')

out         = conv_test.forward(test_img)
grad_out    = np.ones_like(out)
dx          = conv_test.backward(grad_out)

print(f'Conv2D forward  : input {test_img.shape} → output {out.shape}')
print(f'Conv2D backward : grad_out {grad_out.shape} → dx {dx.shape}')
print(f'dx shape sama dengan input: {dx.shape == test_img.shape}')

# test MaxPooling backward
pool_test  = MaxPooling2DLayer(pool_size=(2,2))
out_pool   = pool_test.forward(test_img)
grad_pool  = np.ones_like(out_pool)
dx_pool    = pool_test.backward(grad_pool)
print(f'\nMaxPool forward  : {test_img.shape} → {out_pool.shape}')
print(f'MaxPool backward : {grad_pool.shape} → {dx_pool.shape}')
print(f'dx shape sama dengan input: {dx_pool.shape == test_img.shape}')

# test GlobalAveragePooling backward
gap_test  = GlobalAveragePooling2DLayer()
out_gap   = gap_test.forward(test_img)
grad_gap  = np.ones_like(out_gap)
dx_gap    = gap_test.backward(grad_gap)
print(f'\nGAP forward  : {test_img.shape} → {out_gap.shape}')
print(f'GAP backward : {grad_gap.shape} → {dx_gap.shape}')
print(f'dx shape sama dengan input: {dx_gap.shape == test_img.shape}')